In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
import warnings

In [ ]:
print("Veriler okunuyor") # Verileri okuyoruz.
train_df = pd.read_csv(r"D:\data ai club\train.csv")
test_df = pd.read_csv(r"D:\data ai club\test.csv")

# bk_level = 0 olanları (değerlendirme dışı) eğitimden çıkarıyoruz
train_df = train_df[train_df['bk_level'] > 0].copy()

print("Özellikler üretiliyor...") #Sızıntısız özellik mühendisliği yapıyoruz. Data leak yok.
def create_features(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True)
    df['starttime'] = pd.to_datetime(df['starttime'], format='mixed', utc=True)
    df['gecen_sure'] = (df['timestamp'] - df['starttime']).dt.total_seconds()
    
    vana_listesi = ['fast_dosage_valve', 'slow_dosage_valve', 'kk_dosage_valve',
                    'bk_dosage_valve', 'kk_irtibat_valve', 'bk_irtibat_valve',
                    'kk_bk_common_discharge']
    
    for vana in vana_listesi:
        df[vana] = df[vana].astype(int) if vana in df.columns else 0
            
    df['is_transfer'] = df['commandno'].isin([19, 20]).astype(int)
    df['is_dosage'] = df['commandno'].isin([21, 22]).astype(int)
    df['m243_special'] = ((df['machineid'] == 243) & (df['commandno'] == 20)).astype(int)
    df['dosage_active'] = (df['is_dosage'] & (df['fast_dosage_valve'] | df['slow_dosage_valve'])).astype(int)
    
    # Kümülatif özellikler
    df['fast_valve_cumsum'] = df.groupby('batchkey')['fast_dosage_valve'].cumsum()
    df['slow_valve_cumsum'] = df.groupby('batchkey')['slow_dosage_valve'].cumsum()
    df['discharge_cumsum'] = df.groupby('batchkey')['kk_bk_common_discharge'].cumsum()
    
    # Zaman Pencereleri
    df['fast_valve_roll10'] = df.groupby('batchkey')['fast_dosage_valve'].transform(lambda x: x.rolling(10, min_periods=1).mean())
    df['slow_valve_roll10'] = df.groupby('batchkey')['slow_dosage_valve'].transform(lambda x: x.rolling(10, min_periods=1).mean())
    df['fast_valve_roll30'] = df.groupby('batchkey')['fast_dosage_valve'].transform(lambda x: x.rolling(30, min_periods=1).mean())
    
    # Durum Değişimi
    df['fast_valve_diff'] = df.groupby('batchkey')['fast_dosage_valve'].diff().fillna(0)
    df['slow_valve_diff'] = df.groupby('batchkey')['slow_dosage_valve'].diff().fillna(0)
    df['both_valves_open'] = (df['fast_dosage_valve'] & df['slow_dosage_valve']).astype(int)

    return df

train_df = create_features(train_df)
test_df = create_features(test_df)

In [ ]:
print("GroupKFold ile 5 ayrı model eğitiliyor")

features_list = [
    'commandno', 'bk_target_level', 'machineid', 'is_transfer', 'is_dosage', 
    'fast_dosage_valve', 'slow_dosage_valve', 'm243_special', 'dosage_active', 
    'gecen_sure', 'fast_valve_cumsum', 'slow_valve_cumsum', 'discharge_cumsum',
    'fast_valve_roll10', 'slow_valve_roll10', 'fast_valve_roll30',
    'fast_valve_diff', 'slow_valve_diff', 'both_valves_open'
]

X = train_df[features_list]
y = train_df['bk_level']
groups = train_df['batchkey']

In [ ]:
# 5 farklı gruba bölüyoruz (Aynı batchkey hiçbir zaman hem train hem test içinde olmaz)
gkf = GroupKFold(n_splits=5)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):
    print(f"Fold {fold+1} Eğitiliyor")
    
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
    
    # objective='mae'
    model = lgb.LGBMRegressor(
        objective='mae',         # Yarışma metriğine özel eğitim
        n_estimators=1000,       # Çok daha fazla ağaç
        learning_rate=0.03,      # Daha yavaş ve sağlam öğrenme
        max_depth=8,             # Aşırı ezberlemeyi keser
        random_state=42 + fold,  # Her fold'da hafif farklılık
        n_jobs=-1
    )
    
    # Early Stopping ile model hata yapmaya başlayınca eğitimi kesiyoruz
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

In [ ]:
print("Sonuçlar Kaggle için kaydediliyor")
test_df['Predicted'] = test_preds

submission_df = pd.read_csv(r"D:\data ai club\sample_submission.csv")

if len(test_df) == len(submission_df):
    submission_df['Predicted'] = test_df['Predicted'].values
    submission_df.to_csv("sonuc_submission.csv", index=False)
    print("İŞLEM TAMAM!")
else:
    print("HATA: Satır sayıları tutmuyor!")